# Custom RNN Layer

# 1. Import Libraries

In [4]:
import numpy as np

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 2. Define a Custom RNN Layer

We’ll implement a simple RNN cell:
* Hidden state `h_t = tanh(W_x * x_t + W_h * h_{t-1} + b)`
* Output `y_t = h_t` (or you can add another layer for output)

In [6]:
class CustomRNN(layers.Layer):
    def __init__(self, units, **kwargs):
        super(CustomRNN, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W_x = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='W_x'
        )
        self.W_h = self.add_weight(
            shape=(self.units, self.units),
            initializer='orthogonal',
            trainable=True,
            name='W_h'
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='b'
        )

    def call(self, inputs):
        # inputs: (batch_size, time_steps, input_dim)
        # Unstack along time axis -> list of shape (batch_size, input_dim)
        inputs_unstacked = tf.unstack(inputs, axis=1)
        h = tf.zeros((tf.shape(inputs)[0], self.units))
        outputs = []

        for x_t in inputs_unstacked:
            h = tf.tanh(tf.matmul(x_t, self.W_x) + tf.matmul(h, self.W_h) + self.b)
            outputs.append(h)

        # Stack outputs: (batch_size, time_steps, units)
        return tf.stack(outputs, axis=1)


# 3. Test the Custom RNN

In [7]:
# Create some fake sequential data
x_train = np.random.randn(32, 10, 8).astype(np.float32)  # (batch, time_steps, input_dim)
y_train = np.random.randn(32, 10, 5).astype(np.float32)  # output_dim=5

# Define model
inputs = layers.Input(shape=(10, 8))
rnn_out = CustomRNN(16)(inputs)  # 16 hidden units
outputs = layers.Dense(5)(rnn_out)  # map to output_dim
model = models.Model(inputs, outputs)

# Compile and train
model.compile(optimizer='adam', loss='mse')
model.summary()

model.fit(x_train, y_train, epochs=5)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_rnn_1 (CustomRNN)        │ (None, 10, 16)         │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 5)          │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 485 (1.89 KB)

 Trainable params: 485 (1.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 1.6726
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.6599
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 1.6474
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 1.6351
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 1.6230
